In [1]:
# %%
import pandas as pd 
import numpy as np
# import matplotlib.pyplot as plt 
# import seaborn as sns
import torch
from functools import partial
from openai import AsyncOpenAI

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../"))
if project_root not in sys.path:
    sys.path.append(project_root)

# Load .env from project root
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, ".env"))

from shared_functions.gg_sheet_drive import *

import ast

In [2]:
# Read model config from environment
LLM_MODEL = os.environ["LLM_MODEL"]
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "nomic-ai/nomic-embed-text-v1.5")
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1")
EMBED_BASE_URL = os.environ.get("EMBED_BASE_URL", "http://127.0.0.1:8081/v1")

print(f"LLM_MODEL     : {LLM_MODEL}")
print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")
print(f"LLM_BASE_URL  : {LLM_BASE_URL}")
print(f"EMBED_BASE_URL : {EMBED_BASE_URL}")

import asyncio
from lightrag.utils import setup_logger, EmbeddingFunc
from lightrag.llm.hf import hf_embed
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import gpt_4o_mini_complete, gpt_4o_complete, openai_embed, openai_complete
from lightrag.utils import setup_logger

LLM_MODEL     : Qwen/Qwen2.5-7B-Instruct-AWQ
EMBEDDING_MODEL: nomic-ai/nomic-embed-text-v1.5
LLM_BASE_URL  : http://127.0.0.1:8080/v1
EMBED_BASE_URL : http://127.0.0.1:8081/v1


In [3]:
data_dir = os.path.join(project_root, 'data', 'external', 'medqa')

In [3]:
setup_logger("lightrag", level="INFO")

WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

### Ingestion

In [2]:
async def hf_model_complete(prompt: str, system_prompt=None, history_messages=[], **kwargs):
    device = model.device
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    # Use chat template to wrap the complex LightRAG instructions
    tokenized_chat = llm_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    inputs = {"input_ids": tokenized_chat.to(device)}

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id
    )

    decoded = llm_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return decoded.strip()

In [3]:
async def llm_complete(prompt, system_prompt=None, history_messages=None, **kwargs):
    """LLM completion using the model configured in .env (LLM_MODEL)."""
    kwargs.update({
        "temperature": 0.1,
        "top_p": 0.95,
        "max_tokens": 1024,
    })

    return await openai_complete(
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs
    )

In [4]:
# Using vLLM — model name and URLs come from .env

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=llm_complete,
    llm_model_name=LLM_MODEL,
    llm_model_max_async=4,
    llm_model_kwargs={
        "base_url": LLM_BASE_URL,
        "api_key": "none"
    },
    chunk_token_size=1200,
    entity_extract_max_gleaning=0,
    default_embedding_timeout=120,
    
    embedding_func=EmbeddingFunc(
        embedding_dim=768, 
        max_token_size=8192,
        func=partial(
            openai_embed.func,
            base_url=EMBED_BASE_URL,
            api_key="none",
            model=EMBEDDING_MODEL
        )
    )
)

INFO: [] Created new empty graph file: ./rag_storage/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_chunks.json'} 0 data


In [5]:
await rag.initialize_storages()

INFO: [] Process 908494 KV load full_docs with 0 records
INFO: [] Process 908494 KV load text_chunks with 0 records
INFO: [] Process 908494 KV load full_entities with 0 records
INFO: [] Process 908494 KV load full_relations with 0 records
INFO: [] Process 908494 KV load entity_chunks with 0 records
INFO: [] Process 908494 KV load relation_chunks with 0 records
INFO: [] Process 908494 KV load llm_response_cache with 0 records
INFO: [] Process 908494 doc status load doc_status with 0 records


In [ ]:
sample_textbook = os.path.join(data_dir, 'textbooks', 'Anatomy_Gray.txt')

with open(sample_textbook, 'r') as f:
    text = f.read()

In [10]:
await rag.ainsert(text)

INFO: Created 1 duplicate document records with track_id: insert_20260312_010439_5ed94aa4
INFO: Preserving 1 failed document entries for manual review
INFO: Reset 1 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-21250d6571086cc53d058350568a0609


INFO: Chunk 1 of 468 extracted 30 Ent + 0 Rel chunk-12ebe42e580fd1ab8c7674e6a0142868
INFO: Chunk 2 of 468 extracted 31 Ent + 0 Rel chunk-8c530ec36bf9d30853dccf355f45d595
INFO: Chunk 3 of 468 extracted 28 Ent + 3 Rel chunk-11366959b155e9dcfcd79e0b923979de
INFO: Chunk 4 of 468 extracted 27 Ent + 0 Rel chunk-0bc4d75894db09d737d9d47715131315
INFO: Chunk 5 of 468 extracted 28 Ent + 0 Rel chunk-c8dd6947aa114ed520c7cc1f226401e3
INFO: Chunk 6 of 468 extracted 19 Ent + 7 Rel chunk-2622e0177e14919615609223a054c41f
INFO: Chunk 7 of 468 extracted 25 Ent + 0 Rel chunk-65d8ba7d2670990dd9ac8863c0407af0
INFO: Chunk 8 of 468 extracted 15 Ent + 8 Rel chunk-0d0ff62c79cbf95bf7807d6a60f69b6c
INFO: Chunk 9 of 468 extracted 22 Ent + 4 Rel chunk-fcab2acd6c04fa7081aadd31f6063600
INFO: Chunk 10 of 468 extracted 25 Ent + 0 Rel chunk-6c6de0c22ff2e8ef1ba40e2629a57295
INFO: Chunk 11 of 468 extracted 22 Ent + 2 Rel chunk-1239c16ee834c873f5b3ae9d38c5ff95
INFO: Chunk 12 of 468 extracted 21 Ent + 3 Rel chunk-75fd21d9ae

'insert_20260312_010439_5ed94aa4'

## 🧪 Model Test Block

Use the cells below to test the LLM and embedding servers independently, **without** needing a full RAG pipeline.

#### Unit test

In [11]:
# --- Test: Direct LLM call via OpenAI-compatible API ---
from openai import AsyncOpenAI

llm_client = AsyncOpenAI(base_url=LLM_BASE_URL, api_key="none")

TEST_PROMPT = "What are the main causes of hypertension? Answer in 3 bullet points."
TEST_SYSTEM  = "You are a helpful medical assistant."

response = await llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": TEST_SYSTEM},
        {"role": "user",   "content": TEST_PROMPT},
    ],
    temperature=0.1,
    max_tokens=512,
)

print(f"Model  : {response.model}")
print(f"Tokens : prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")
print("\n--- Response ---")
print(response.choices[0].message.content)

Model  : Qwen/Qwen2.5-7B-Instruct-AWQ
Tokens : prompt=35, completion=84

--- Response ---
- **Genetic Factors**: A family history of hypertension can increase an individual's risk.
- **Lifestyle Choices**: Poor diet (high in sodium and low in potassium), lack of physical activity, obesity, and excessive alcohol consumption contribute to hypertension.
- **Age and Gender**: Risk increases with age, and men are more likely to develop hypertension than women until menopause, after which the risk for women rises.


In [ ]:
# --- Test: Embedding server ---
embed_client = AsyncOpenAI(base_url=EMBED_BASE_URL, api_key="none")

TEST_TEXT = "The heart pumps blood throughout the body."

embed_response = await embed_client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=TEST_TEXT,
)

vec = embed_response.data[0].embedding
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Vector dimension: {len(vec)}")
print(f"First 8 values  : {vec[:8]}")

Embedding model : nomic-ai/nomic-embed-text-v1.5
Vector dimension: 768
First 8 values  : [1.65087890625, 1.772070288658142, -4.043164253234863, 0.4876464903354645, 0.6470062136650085, 0.508770763874054, -0.3969482481479645, 1.058740258216858]


In [16]:
# --- Test: RAG query (requires storages to be initialized and documents inserted) ---
QUERY = "What is the function of the aorta?"
MODE  = "local"  # options: naive | local | global | hybrid

result = await rag.aquery(QUERY, param=QueryParam(mode=MODE))
print(f"Query : {QUERY}")
print(f"Mode  : {MODE}")
print("\n--- Answer ---")
print(result)

INFO:  == LLM cache == saving: local:keywords:2f9f2453fc68080144ef03235d72eaad
INFO: Query nodes: Blood transport, Arterial system, Cardiovascular function (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 18 relations
INFO: Raw search results: 40 entities, 18 relations, 0 vector chunks
INFO: After truncation: 40 entities, 18 relations
INFO: Selecting 85 from 85 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 18 relations
INFO: Round-robin merged chunks: 85 -> 85 (deduplicated 0)
INFO: Final context: 40 entities, 18 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E3/5 E1/6 E2/7 E2/8 E1/9 E1/10 E2/11 E2/12 E1/13 E1/14 E3/15 E1/16 E3/17 E2/18 E1/19 E1/20
ERROR: Query failed: Error code: 400 - {'error': {'message': "You passed 7169 input tokens and requested 1024 output tokens. However, the model's context length is only 8192 tokens, resulting in a maximum input length of 7168 tokens. Please reduce the length of the 

Query : What is the function of the aorta?
Mode  : local

--- Answer ---
None


#### QA Test

In [4]:
from test_full_pipeline import BioMed_RAG

bio_hehe = BioMed_RAG()

LLM_MODEL     : Qwen/Qwen2.5-7B-Instruct-AWQ
EMBEDDING_MODEL: nomic-ai/nomic-embed-text-v1.5
LLM_BASE_URL  : http://127.0.0.1:8080/v1
EMBED_BASE_URL : http://127.0.0.1:8081/v1


In [5]:
qa_path = os.path.join(data_dir, 'medqa_form1.csv')

medqa = pd.read_csv(qa_path, index_col = 0)

medqa = medqa.drop_duplicates(subset = 'question')

In [6]:
unqa = medqa[medqa['processed'] == 0].copy()

unqa['answer'] = unqa.apply(
    lambda row: ast.literal_eval(row['options'])[row['answer']],
    axis=1
)

new_qa = pd.concat([medqa[medqa['processed'] == 1], unqa], axis = 0)

new_qa = new_qa[['question', 'answer']]

In [7]:
new_qa

,question,answer
0,A 23-year-old pregnant woman at 22 weeks gesta...,Nitrofurantoin
1,A 3-month-old baby died suddenly at night whil...,Placing the infant in a supine position on a f...
2,A mother brings her 3-week-old infant to the p...,Abnormal migration of ventral pancreatic bud
3,A pulmonary autopsy specimen from a 58-year-ol...,Thromboembolism
4,A 20-year-old woman presents with menorrhagia ...,Von Willebrand disease
...,...,...
14328,A 75-year-old man presents to his physician af...,Hepatotoxicity
14341,The brain of a 75-year-old man is reviewed at ...,Cytoplasmic inclusions of tau protein
14348,A 55-year-old male presents with a low grade f...,Coccidioidomycosis
14354,A 36-year-old man working as a hospital techni...,Mutation of cell wall D-ala-D-ala to D-ala-D-lac


In [8]:
sample_qa = new_qa.head(5).copy()

sample_qa['llm_answer'] = ''

for i in range(sample_qa.shape[0]):
    sample_qa.loc[i, 'llm_answer'], sample_qa.loc[i, 'retrieved_context'] = await bio_hehe.biomedical_rag(sample_qa.loc[i, 'question'])

INFO: [] Loaded graph from ./rag_storage/graph_chunk_entity_relation.graphml with 5252 nodes, 1000 edges
INFO:nano-vectordb:Load (5252, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_entities.json'} 5252 data
INFO:nano-vectordb:Load (1000, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_relationships.json'} 1000 data
INFO:nano-vectordb:Load (468, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_chunks.json'} 468 data
INFO: [] Process 1136714 KV load full_docs with 1 records
INFO: [] Process 1136714 KV load text_chunks with 468 records
INFO: [] Process 1136714 KV load full_entities with 1 records
INFO: [] Process 1136714 KV load full_relations with 1 records
INFO: [] Process 1136714 KV load entity_chunks with 5252 records
INFO: [] Process 1136714 KV load relation_chunks with 1000 records
INFO: [] P

In [9]:
sample_qa

,question,answer,llm_answer,retrieved_context
0,A 23-year-old pregnant woman at 22 weeks gesta...,Nitrofurantoin,The retrieved context does not contain this in...,\nKnowledge Graph Data (Entity):\n\n```json\n{...
1,A 3-month-old baby died suddenly at night whil...,Placing the infant in a supine position on a f...,Place the baby on its back to sleep.,\nKnowledge Graph Data (Entity):\n\n```json\n{...
2,A mother brings her 3-week-old infant to the p...,Abnormal migration of ventral pancreatic bud,Gastroesophageal Reflux\n\nThis condition can ...,\nKnowledge Graph Data (Entity):\n\n```json\n{...
3,A pulmonary autopsy specimen from a 58-year-ol...,Thromboembolism,Fibrosis of the pulmonary artery lumen suggest...,\nKnowledge Graph Data (Entity):\n\n```json\n{...
4,A 20-year-old woman presents with menorrhagia ...,Von Willebrand disease,The retrieved context does not contain this in...,\nKnowledge Graph Data (Entity):\n\n```json\n{...


In [ ]:
# write_df_to_gs(sample_qa, 'Qwen2.5-7B-Instruct-AWQ_nomic-embed-text-v1.5')

'DataFrame appended to existing Google Sheet tab: Qwen2.5-7B-Instruct-AWQ_nomic-embed-text-v1.5'

## Evaluation